# Honeydew AI — расширенное облачное обучение

Цель — не менее 200 000 обучающих примеров: 10 000 оригинальных синтетических композиций + до 190 000 MIDI из PDMX (только Public Domain/CC0 без конфликта лицензии и с доступным MIDI). Скрипт строго остановится, если не удастся собрать все 190 000 внешних файлов: не подменяем недостающие композиции дубликатами. PDMX MIDI-архив около 214.4 MB; CSV около 225.4 MB. Checkpoint сохраняется на Google Drive после каждой эпохи.


In [ ]:
!nvidia-smi || true
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


In [ ]:
!git clone --depth 1 --branch ai-beatmaker https://github.com/djdhsvsbdb-design/honeydew.git
%cd honeydew
!python -m pip install -r tools/ai/requirements.txt


In [ ]:
# Скачать и проверить до 190 000 MIDI PDMX; собрать не менее 200 000 примеров суммарно
!python tools/ai/download_pdmx.py --limit 190000
!python tools/ai/build_dataset.py --synthetic-songs 10000 --external-limit 190000 --external-manifest data/ai/external/pdmx/external-manifest.jsonl --out data/ai/honeydew-200k.jsonl
!wc -l data/ai/honeydew-200k.jsonl


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/Honeydew')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = CHECKPOINT_DIR / 'honeydew-music.pt'
print('Checkpoint:', CHECKPOINT)


In [ ]:
# Обучение на наборе 200k; checkpoint записывается после каждой эпохи
!python tools/ai/train_music.py --data data/ai/honeydew-200k.jsonl --epochs 20 --batch-size 8 --block 256 --val-fraction 0.1 --out "{CHECKPOINT}" --resume


In [ ]:
!python tools/ai/generate_music.py --checkpoint /content/drive/MyDrive/Honeydew/honeydew-music.best.pt --style trap --bars 8 --bpm 145 --out /content/honeydew-generated


In [ ]:
from google.colab import files
from pathlib import Path

for filename in ["/content/drive/MyDrive/Honeydew/honeydew-music.pt", "/content/drive/MyDrive/Honeydew/honeydew-music.best.pt", "/content/honeydew-generated.mid", "/content/honeydew-generated.json"]:
    if Path(filename).exists():
        files.download(filename)
    else:
        print(f"Пока нет файла: {filename}")
